The purpose of this notebook is to generate a stratification frame like the ones for Sweden and Denmark, but for the Netherlands.

As such, it relies on CBS data to build the exhaustive frame.

In [2]:
import requests
import pandas as pd

In [9]:
import requests
import pandas as pd
import time

BASE = "https://opendata.cbs.nl/ODataApi/OData/85525NED"

TARGET_PERIOD = "2024JJ00"
GEO_PREFIX = "GM"   # use "GM" for gemeenten; if that yields 0 rows, try "CR"

EDU_MAP = {
    "k_11Basisonderwijs_5": "Basisonderwijs",
    "k_12VmboHavoVwoOnderbouwMbo1_6": "Vmbo, havo-/vwo-onderbouw, mbo1",
    "k_21HavoVwoMbo24_7": "Havo, vwo, mbo2-4",
    "k_31HboWoBachelor_8": "Hbo-, wo-bachelor",
    "k_32HboWoMasterDoctor_9": "Hbo-, wo-master, doctor",
}

def fetch_json(url: str, timeout: int = 60) -> dict:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()

def fetch_dimension(name: str) -> pd.DataFrame:
    js = fetch_json(f"{BASE}/{name}")
    return pd.DataFrame(js["value"])

# -------------------------
# 1) Load dimensions
# -------------------------
regios = fetch_dimension("RegioS")[["Key", "Title"]].rename(
    columns={"Key": "RegioS", "Title": "region_name"}
)

geslacht = fetch_dimension("Geslacht")[["Key", "Title"]].rename(
    columns={"Key": "Geslacht", "Title": "gender"}
)

leeftijd = fetch_dimension("Leeftijd")[["Key", "Title"]].rename(
    columns={"Key": "Leeftijd", "Title": "age_group"}
)

# Men / women only
sex_keys = geslacht.loc[
    geslacht["gender"].isin(["Mannen", "Vrouwen"]),
    "Geslacht"
].tolist()

# Keep detailed age groups only, not totals
# This is robust to label wording differences.
age_keys = leeftijd.loc[
    ~leeftijd["age_group"].str.contains("15 tot 75 jaar|Totaal", case=False, na=False),
    "Leeftijd"
].tolist()

print(f"Found {len(regios)} regions in RegioS")
print(f"Using geo prefix: {GEO_PREFIX}")
print(f"Using {len(sex_keys)} sex categories and {len(age_keys)} age groups")

# -------------------------
# 2) Query one sex x age slice at a time
#    to avoid the CBS 10k-row query limit
# -------------------------
def download_slice(period_key: str, sex_key: str, age_key: str, geo_prefix: str) -> pd.DataFrame:
    select_cols = ",".join(
        ["RegioS", "Geslacht", "Leeftijd", "Perioden", "Bevolking_1"] + list(EDU_MAP.keys())
    )

    url = (
        f"{BASE}/TypedDataSet"
        f"?$select={select_cols}"
        f"&$filter=(Perioden eq '{period_key}')"
        f" and (Geslacht eq '{sex_key}')"
        f" and (Leeftijd eq '{age_key}')"
        f" and startswith(RegioS,'{geo_prefix}')"
    )

    js = fetch_json(url)
    return pd.DataFrame(js.get("value", []))

parts = []
for s in sex_keys:
    for a in age_keys:
        print(f"Downloading period={TARGET_PERIOD}, sex={s}, age={a}, geo={GEO_PREFIX}")
        part = download_slice(TARGET_PERIOD, s, a, GEO_PREFIX)
        if not part.empty:
            parts.append(part)
        time.sleep(0.2)

if not parts:
    raise RuntimeError(
        f"No rows returned for GEO_PREFIX='{GEO_PREFIX}'. "
        f"If you used 'GM', the OData endpoint may not expose municipalities for this table."
    )

df = pd.concat(parts, ignore_index=True)

# -------------------------
# 3) Attach labels
# -------------------------
df = df.merge(regios, on="RegioS", how="left")
df = df.merge(geslacht, on="Geslacht", how="left")
df = df.merge(leeftijd, on="Leeftijd", how="left")

geo_col_name = "municipality" if GEO_PREFIX == "GM" else "region"
df = df.rename(columns={"region_name": geo_col_name})

# -------------------------
# 4) Reshape education columns
# -------------------------
long = df.melt(
    id_vars=[geo_col_name, "gender", "age_group", "Bevolking_1"],
    value_vars=list(EDU_MAP.keys()),
    var_name="education_key",
    value_name="pct"
)

long["education"] = long["education_key"].map(EDU_MAP)

# Convert percentage -> count
long["N"] = (long["Bevolking_1"] * long["pct"] / 100.0).round().astype("Int64")

# -------------------------
# 5) Final output
# -------------------------
result = long[[geo_col_name, "gender", "age_group", "education", "N"]].copy()
result = result.sort_values([geo_col_name, "gender", "age_group", "education"]).reset_index(drop=True)

outfile = f"85525NED_{GEO_PREFIX}_gender_age_education_N.csv"
result.to_csv(outfile, index=False)

print("\nDone.")
print(f"Rows written: {len(result):,}")
print(f"Output file: {outfile}")
print(result.head(20).to_string(index=False))

Found 524 regions in RegioS
Using geo prefix: GM
Using 2 sex categories and 6 age groups

Done.
Rows written: 25,920
Output file: 85525NED_GM_gender_age_education_N.csv
            municipality gender      age_group                       education     N
's-Gravenhage (gemeente) Mannen 15 tot 25 jaar                  Basisonderwijs  5490
's-Gravenhage (gemeente) Mannen 15 tot 25 jaar               Havo, vwo, mbo2-4 14677
's-Gravenhage (gemeente) Mannen 15 tot 25 jaar               Hbo-, wo-bachelor  3221
's-Gravenhage (gemeente) Mannen 15 tot 25 jaar         Hbo-, wo-master, doctor   512
's-Gravenhage (gemeente) Mannen 15 tot 25 jaar Vmbo, havo-/vwo-onderbouw, mbo1 12700
's-Gravenhage (gemeente) Mannen 25 tot 35 jaar                  Basisonderwijs  3174
's-Gravenhage (gemeente) Mannen 25 tot 35 jaar               Havo, vwo, mbo2-4 15913
's-Gravenhage (gemeente) Mannen 25 tot 35 jaar               Hbo-, wo-bachelor  9387
's-Gravenhage (gemeente) Mannen 25 tot 35 jaar         Hbo-, wo-ma

In [11]:
import re
import unicodedata
import requests
import pandas as pd

# -----------------------------
# Settings
# -----------------------------
INPUT_FILE = "85525NED_GM_gender_age_education_N.csv"   # change if needed
OUTPUT_FILE = "85525NED_GM_gender_age_education_N_clean.csv"

# Official municipality lookup source
MUNI_TABLE = "85755NED"
BASE = f"https://opendata.cbs.nl/ODataApi/OData/{MUNI_TABLE}"


# -----------------------------
# Helpers
# -----------------------------
def fetch_all(url: str) -> pd.DataFrame:
    rows = []
    while url:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        js = r.json()
        rows.extend(js.get("value", []))
        url = js.get("odata.nextLink") or js.get("@odata.nextLink")
    return pd.DataFrame(rows)


def normalize_text(x: str) -> str:
    """
    Normalize municipality names so matching is robust.
    """
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()

    # Unicode normalize
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))

    # Common punctuation variants
    x = x.replace("’", "'").replace("`", "'").replace("´", "'")
    x = x.replace("-", " ")
    x = x.replace("/", " ")

    # Remove bracketed clarifiers like "(gemeente)"
    x = re.sub(r"\(.*?\)", "", x)

    # Remove extra whitespace
    x = re.sub(r"\s+", " ", x).strip()

    return x


def canonicalize_municipality_name(x: str) -> str:
    """
    Extra municipality-specific cleanup rules.
    """
    x = normalize_text(x)

    replacements = {
        "groningen gemeente": "groningen",
        "s gravenhage": "'s gravenhage",
        "den haag": "'s gravenhage",   # optional, for matching alternate names
    }

    return replacements.get(x, x)


# -----------------------------
# 1) Read your result file
# -----------------------------
result = pd.read_csv(INPUT_FILE)

if "municipality" not in result.columns:
    raise ValueError("Expected a 'municipality' column in the input file.")

result["municipality_raw"] = result["municipality"].astype(str)
result["municipality_clean"] = result["municipality_raw"].apply(canonicalize_municipality_name)


# -----------------------------
# 2) Download official municipality list from CBS
# -----------------------------
# We use RegioS to get GM codes and official labels
regios = fetch_all(f"{BASE}/RegioS")[["Key", "Title"]].rename(
    columns={"Key": "gm_code", "Title": "municipality_official"}
)

# Keep municipalities only
municipalities = regios[regios["gm_code"].str.startswith("GM")].copy()

municipalities["municipality_clean"] = municipalities["municipality_official"].apply(
    canonicalize_municipality_name
)

# In case of duplicates after cleaning, keep the first official form
municipalities = municipalities.drop_duplicates(subset=["municipality_clean"]).copy()


# -----------------------------
# 3) Match your result to official municipality names
# -----------------------------
cleaned = result.merge(
    municipalities[["gm_code", "municipality_official", "municipality_clean"]],
    on="municipality_clean",
    how="left"
)

# Flag matched vs unmatched
cleaned["is_valid_municipality"] = cleaned["gm_code"].notna()

matched = cleaned[cleaned["is_valid_municipality"]].copy()
unmatched = cleaned[~cleaned["is_valid_municipality"]].copy()

# Replace municipality with official CBS name
matched["municipality"] = matched["municipality_official"]


# -----------------------------
# 4) Final cleaned output
# -----------------------------
final_cols = [c for c in ["gm_code", "municipality", "gender", "age_group", "education", "N"] if c in matched.columns]
matched = matched[final_cols].copy()

matched.to_csv(OUTPUT_FILE, index=False)

print(f"Input rows:   {len(result):,}")
print(f"Matched rows: {len(matched):,}")
print(f"Unmatched rows: {len(unmatched):,}")
print(f"Saved cleaned file to: {OUTPUT_FILE}")

if len(unmatched) > 0:
    print("\nUnmatched municipality values:")
    print(unmatched["municipality_raw"].drop_duplicates().sort_values().to_string(index=False))

Input rows:   25,920
Matched rows: 20,520
Unmatched rows: 5,400
Saved cleaned file to: 85525NED_GM_gender_age_education_N_clean.csv

Unmatched municipality values:
                       Aalburg
                    Appingedam
                         Bedum
                      Beemster
                  Bellingwedde
                   Bergambacht
                      Bernisse
                    Binnenmaas
                  Boarnsterhim
                       Boskoop
                       Boxmeer
                       Brielle
                        Bussum
                   Cromstrijen
                         Cuijk
               De Friese Meren
                      De Marne
                      Delfzijl
                   Dongeradeel
                      Eemsmond
                Ferwerderadiel
                 Franekeradeel
              Gaasterlân-Sleat
                  Geldermalsen
                 Giessenlanden
                 Graft-De Rijp
                         Grave

In [ ]:
matched.head()

,gm_code,municipality,gender,age_group,education,N
0,GM0518,'s-Gravenhage (gemeente),Mannen,15 tot 25 jaar,Basisonderwijs,5490.0
1,GM0518,'s-Gravenhage (gemeente),Mannen,15 tot 25 jaar,"Havo, vwo, mbo2-4",14677.0
2,GM0518,'s-Gravenhage (gemeente),Mannen,15 tot 25 jaar,"Hbo-, wo-bachelor",3221.0
3,GM0518,'s-Gravenhage (gemeente),Mannen,15 tot 25 jaar,"Hbo-, wo-master, doctor",512.0
4,GM0518,'s-Gravenhage (gemeente),Mannen,15 tot 25 jaar,"Vmbo, havo-/vwo-onderbouw, mbo1",12700.0
...,...,...,...,...,...,...
25855,GM0193,Zwolle,Vrouwen,65 tot 75 jaar,Basisonderwijs,800.0
25856,GM0193,Zwolle,Vrouwen,65 tot 75 jaar,"Havo, vwo, mbo2-4",2259.0
25857,GM0193,Zwolle,Vrouwen,65 tot 75 jaar,"Hbo-, wo-bachelor",1197.0
25858,GM0193,Zwolle,Vrouwen,65 tot 75 jaar,"Hbo-, wo-master, doctor",506.0


In [12]:
df['N'].describe()

count    110808.000000
mean        648.193271
std        6396.874281
min           0.000000
25%          84.000000
50%         187.000000
75%         364.000000
max      931298.000000
Name: N, dtype: float64

In [14]:
# Keep only municipalities that have at least one non-NaN value in N
df = df[df.groupby("municipality")["N"].transform(lambda s: s.notna().any())].reset_index(drop=True)

print(df.shape)

(110808, 4)


In [21]:
df = df[df['gender'] != 'Totaal mannen en vrouwen']
df = df[df['age'] != 'Totaal']

In [22]:
df.sample(10)

,municipality,gender,age,N
53084,Leidschendam-Voorburg,Vrouwen,54 jaar,558.0
32310,Gemert-Bakel,Vrouwen,2 jaar,191.0
16560,Brunssum,Mannen,36 jaar,146.0
44324,Hoeksche Waard,Vrouwen,43 jaar,541.0
34246,Gooise Meren,Vrouwen,12 jaar,393.0
67688,Oldambt,Vrouwen,76 jaar,241.0
1188,Aalsmeer,Vrouwen,0 jaar,114.0
15653,Breda,Mannen,95 jaar,31.0
31320,Etten-Leur,Vrouwen,0 jaar,184.0
90389,Tytsjerksteradiel,Vrouwen,95 jaar,12.0


In [23]:
print(df['N'].sum())

17969558.0
